In [22]:
from credenciales import SMTP_HOST, USERNAME, PASSWORD
import smtplib
from email.mime.text import MIMEText

port = 2525   # prueba este primero (permite sin TLS / STARTTLS)

msg = MIMEText("Esto es solo una prueba rápida después del registro.\nDebería aparecer en Mailtrap.")
msg['Subject'] = 'Test Práctica Redes - SMTP sin TLS'
msg['From'] = 'candela.naya.lopez@udc.es'
msg['To'] = 'destino@prueba.com'

try:
    with smtplib.SMTP(SMTP_HOST, port, timeout=10) as server:
        server.login(USERNAME, PASSWORD)
        server.send_message(msg)
    print("¡ÉXITO! Revisa tu sandbox en mailtrap.io → debería aparecer el correo en segundos.")
except Exception as e:
    print(e)
    print("\nPrueba cambiando port=25 o port=587 y añadiendo server.starttls() después de la conexión.")

¡ÉXITO! Revisa tu sandbox en mailtrap.io → debería aparecer el correo en segundos.


In [26]:
# Tarea 2.1 – SMTP sin TLS (todo en claro para Wireshark)
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from credenciales import SMTP_HOST, USERNAME, PASSWORD

port = 2525          # Alternativa: prueba 25 si 2525 no deja sin cifrado
sender_email = "candela.naya.lopez@udc.es"
receiver_email = "destino@prueba.com"

msg = MIMEMultipart("alternative")
msg["Subject"] = "Práctica 3 – SMTP SIN TLS (en oscuro)"
msg["From"] = sender_email
msg["To"] = receiver_email

texto_plano = "Este correo se envió SIN cifrado TLS.\nTodo (usuario, pass, contenido) se ve en Wireshark."
html = """<html>
  <body>
    <h2>SMTP SIN TLS</h2>
    <p>Este mensaje se envió sin cifrado para demostrar que el protocolo y contenido viajan en claro.</p>
  </body>
</html>"""

msg.attach(MIMEText(texto_plano, "plain"))
msg.attach(MIMEText(html, "html"))

try:
    server = smtplib.SMTP(SMTP_HOST, port, timeout=15)
    server.login(USERNAME, PASSWORD)
    server.send_message(msg)
    server.quit()
    print(f"ÉXITO: Correo SIN TLS enviado a través de {SMTP_HOST}:{port}")
    print("→ Ve a tu sandbox en Mailtrap y comprueba que llegó.")
except Exception as e:
    print("Error al enviar SIN TLS:")
    print(e)
    print("\nConsejo: prueba cambiar port=25 si falla")

ÉXITO: Correo SIN TLS enviado a través de sandbox.smtp.mailtrap.io:2525
→ Ve a tu sandbox en Mailtrap y comprueba que llegó.


In [7]:
# Tarea 2.2 – SMTP con TLS (STARTTLS en puerto 587 – lo más recomendado)
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from credenciales import SMTP_HOST, USERNAME, PASSWORD

port = 587           # Puerto estándar para STARTTLS

sender_email = "candela.naya.lopez@udc.es"
receiver_email = "destino@prueba.com"

msg = MIMEMultipart("alternative")
msg["Subject"] = "Práctica 3 – SMTP CON TLS (cifrado)"
msg["From"] = sender_email
msg["To"] = receiver_email

texto_plano = "Este correo se envió CON cifrado TLS (STARTTLS).\nEl contenido NO se ve en Wireshark después del handshake."
html = """<html>
  <body>
    <h2>SMTP CON TLS</h2>
    <p>Conexión iniciada en plano, pero actualizada a TLS con starttls().</p>
  </body>
</html>"""

msg.attach(MIMEText(texto_plano, "plain"))
msg.attach(MIMEText(html, "html"))

try:
    server = smtplib.SMTP(SMTP_HOST, port, timeout=15)
    server.ehlo()             # Obligatorio antes de starttls en algunos servidores
    server.starttls()         # ← Aquí se activa el cifrado TLS
    server.ehlo()             # Recomendado después
    server.login(USERNAME, PASSWORD)
    server.send_message(msg)
    server.quit()
    print(f"ÉXITO: Correo CON TLS (STARTTLS) enviado a través de {SMTP_HOST}:{port}")
    print("→ Comprueba en Mailtrap.")
except Exception as e:
    print("Error al enviar CON TLS:")
    print(e)
    print("\nAlternativa: prueba puerto 465 + smtplib.SMTP_SSL(...) si falla")

ÉXITO: Correo CON TLS (STARTTLS) enviado a través de sandbox.smtp.mailtrap.io:587
→ Comprueba en Mailtrap.


In [27]:
# Tarea 3.1 – POP3 sin TLS (puerto 110 – texto claro para Wireshark)
import poplib
from credenciales import USERNAME, PASSWORD
from credenciales import POP3_HOST  # Si no están, pon directamente: "pop3.mailtrap.io", 110

POP3_PORT = 1100
try:
    server = poplib.POP3(POP3_HOST, POP3_PORT, timeout=15)
    server.set_debuglevel(1)  # Muestra comandos POP3 en consola → útil para ver el protocolo en claro
    
    print("Conectando a POP3 sin TLS...")
    server.user(USERNAME)
    server.pass_(PASSWORD)
    
    num_messages = len(server.list()[1])
    print(f"ÉXITO: Conexión POP3 sin TLS establecida.")
    print(f"Mensajes en el buzón: {num_messages}")
    
    if num_messages > 0:
        print("\nMostrando el ÚLTIMO mensaje (retr en claro):")
        response, lines, octets = server.retr(num_messages)
        print("-" * 60)
        for line in lines:
            try:
                print(line.decode('utf-8', errors='replace'))
            except:
                print(line)
        print("-" * 60)
    else:
        print("No hay mensajes en el buzón todavía. Envía uno con SMTP primero.")
    
    server.quit()
    print("Sesión POP3 cerrada correctamente.")
    print("→ Captura Wireshark ahora (filtro: pop) → verás USER/PASS y contenido en plano.")

except Exception as e:
    print("Error en POP3 sin TLS:")
    print(e)
    print("\nConsejos:")
    print("- Asegúrate de haber enviado al menos un correo con SMTP antes.")
    print("- Verifica USERNAME y PASSWORD en credenciales.py")
    print("- Prueba aumentar timeout si es muy lento.")

Conectando a POP3 sin TLS...
*cmd* 'USER 6ab5ebe2bc002c'
*cmd* 'PASS a5eab68feb719f'
*cmd* 'LIST'
ÉXITO: Conexión POP3 sin TLS establecida.
Mensajes en el buzón: 10

Mostrando el ÚLTIMO mensaje (retr en claro):
*cmd* 'RETR 10'
------------------------------------------------------------
Content-Type: text/plain; charset="utf-8"
MIME-Version: 1.0
Content-Transfer-Encoding: base64
Subject: =?utf-8?q?Test_Pr=C3=A1ctica_Redes_-_SMTP_sin_TLS?=
From: candela.naya.lopez@udc.es
To: destino@prueba.com

RXN0byBlcyBzb2xvIHVuYSBwcnVlYmEgcsOhcGlkYSBkZXNwdcOpcyBkZWwgcmVnaXN0cm8uCkRl
YmVyw61hIGFwYXJlY2VyIGVuIE1haWx0cmFwLg==

------------------------------------------------------------
*cmd* 'QUIT'
Sesión POP3 cerrada correctamente.
→ Captura Wireshark ahora (filtro: pop) → verás USER/PASS y contenido en plano.


In [25]:
# Tarea 3.2 – POP3 con TLS (STLS en puerto 110 – usando ssl para cifrado)
import poplib
import ssl
from credenciales import USERNAME, PASSWORD
from credenciales import POP3_HOST  # "pop3.mailtrap.io", 110

POP3_PORT = 9950

try:
    server = poplib.POP3(POP3_HOST, POP3_PORT, timeout=15)
    server.set_debuglevel(1)  # Muestra comandos → verás +OK hasta STLS, luego cifrado
    
    print("Conectando a POP3 (inicio en plano)...")
    
    # Activar TLS explícito (STARTTLS equivalente para POP3 → se llama STLS)
    context = ssl.create_default_context()
    server.stls(context)
    
    print("TLS activado con STLS → a partir de aquí el tráfico está cifrado.")
    
    server.user(USERNAME)
    server.pass_(PASSWORD)
    
    num_messages = len(server.list()[1])
    print(f"ÉXITO: Conexión POP3 con TLS establecida.")
    print(f"Mensajes en el buzón: {num_messages}")
    
    if num_messages > 0:
        print("\nMostrando el ÚLTIMO mensaje (tras cifrado):")
        response, lines, octets = server.retr(num_messages)
        print("-" * 60)
        for line in lines:
            try:
                print(line.decode('utf-8', errors='replace'))
            except:
                print(line)
        print("-" * 60)
    else:
        print("No hay mensajes. Envía uno con SMTP primero.")
    
    server.quit()
    print("Sesión POP3 con TLS cerrada correctamente.")
    print("→ En Wireshark verás que después de STLS todo está cifrado (no legible).")

except Exception as e:
    print("Error en POP3 con TLS:")
    print(e)
    print("\nConsejos:")
    print("- Asegúrate de llamar server.stls() ANTES de user() y pass_()")
    print("- Si falla STLS, prueba con poplib.POP3_SSL(POP3_HOST, 995) en su lugar (aunque en Mailtrap sandbox suele ser 110+STLS)")
    print("- Verifica credenciales y que hayas enviado correos antes.")

Conectando a POP3 (inicio en plano)...
*cmd* 'CAPA'
*cmd* 'STLS'
TLS activado con STLS → a partir de aquí el tráfico está cifrado.
*cmd* 'USER 6ab5ebe2bc002c'
*cmd* 'PASS a5eab68feb719f'
*cmd* 'LIST'
ÉXITO: Conexión POP3 con TLS establecida.
Mensajes en el buzón: 10

Mostrando el ÚLTIMO mensaje (tras cifrado):
*cmd* 'RETR 10'
------------------------------------------------------------
Content-Type: multipart/alternative;
 boundary="===============1651097619716821150=="
MIME-Version: 1.0
Subject: =?utf-8?q?Pr=C3=A1ctica_3_=E2=80=93_SMTP_CON_TLS_=28cifrado=29?=
From: alumno@practica.redes
To: destino@prueba.com

--===============1651097619716821150==
Content-Type: text/plain; charset="utf-8"
MIME-Version: 1.0
Content-Transfer-Encoding: base64

RXN0ZSBjb3JyZW8gc2UgZW52acOzIENPTiBjaWZyYWRvIFRMUyAoU1RBUlRUTFMpLgpFbCBjb250
ZW5pZG8gTk8gc2UgdmUgZW4gV2lyZXNoYXJrIGRlc3B1w6lzIGRlbCBoYW5kc2hha2Uu

--===============1651097619716821150==
Content-Type: text/html; charset="utf-8"
MIME-Version: 1.0
Co